In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from pathlib import Path
from PIL import Image
import pandas as pd
import os


def setup_center_crop_transform(img_size=224):
    """Transform used for validation / test."""
    return transforms.Compose(
        [
            transforms.Resize(int(img_size * 1.14)),
            transforms.CenterCrop(img_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
    )


def setup_train_transform(img_size=224):
    """Slightly richer augmentation for training."""
    return transforms.Compose(
        [
            transforms.RandomResizedCrop(img_size, scale=(0.8, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
    )


def resolve_train_path(root_path: Path) -> Path:
    """
    Return the directory that directly contains both 'cat' and 'dog' sub‑folders.
    Searches recursively from ``root_path`` and returns the first match.
    """
    # First check the most common locations
    candidates = [
        root_path / "train",
        root_path / "train" / "train",
    ]
    for cand in candidates:
        if (cand / "cat").is_dir() and (cand / "dog").is_dir():
            return cand

    # Fallback: recursive search
    for d in root_path.rglob("*"):
        if d.is_dir() and (d / "cat").is_dir() and (d / "dog").is_dir():
            return d
    raise FileNotFoundError(f"Could not locate training images under {root_path}")


def get_train_dir(base_path: Path) -> Path:
    """
    Wrapper that guarantees the returned path contains the 'cat' and 'dog' folders.
    If the resolved path is one level above, we descend automatically.
    """
    path = resolve_train_path(base_path)
    if not ((path / "cat").is_dir() and (path / "dog").is_dir()):
        deeper = path / "train"
        if (deeper / "cat").is_dir() and (deeper / "dog").is_dir():
            path = deeper
    return path


def setup_train_val_loaders(data_dir, batch_size=64, dryrun=False, val_frac=0.2):
    """Create train/validation loaders from the ImageFolder structure."""
    train_path = get_train_dir(Path(data_dir))
    train_transform = setup_train_transform()
    val_transform = setup_center_crop_transform()
    full_dataset = ImageFolder(root=train_path, transform=train_transform)

    val_size = int(len(full_dataset) * val_frac)
    train_size = len(full_dataset) - val_size
    train_set, val_set = random_split(
        full_dataset,
        [train_size, val_size],
        generator=torch.Generator().manual_seed(42),
    )

    if dryrun:
        train_set = Subset(train_set, range(min(200, len(train_set))))
        val_set = Subset(val_set, range(min(100, len(val_set))))

    val_set.dataset.transform = val_transform

    train_loader = DataLoader(
        train_set, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True
    )
    val_loader = DataLoader(
        val_set, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True
    )
    return train_loader, val_loader


class TestImageDataset(torch.utils.data.Dataset):
    """Dataset that loads test images and keeps their original filenames."""

    def __init__(self, paths, transform=None):
        self.paths = sorted(paths)
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img_path = self.paths[idx]
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img


def find_all_image_files(root_path: Path):
    """Recursively find all JPEG/PNG images under root_path."""
    return [
        p
        for p in root_path.rglob("*")
        if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    ]


def locate_test_root(base_path: Path) -> Path:
    """
    Locate the directory that holds the test images.
    Preference is given to a folder named 'test' that does **not**
    contain the 'cat' or 'dog' sub‑folders.
    """
    test_candidate = base_path / "test"
    if test_candidate.is_dir():
        if not ((test_candidate / "cat").is_dir() or (test_candidate / "dog").is_dir()):
            if find_all_image_files(test_candidate):
                return test_candidate

    for d in base_path.rglob("*"):
        if d.is_dir() and not ((d / "cat").is_dir() or (d / "dog").is_dir()):
            if find_all_image_files(d):
                return d
    raise FileNotFoundError(f"Could not locate test images under {base_path}")


def setup_test_loader(data_dir, batch_size=64, dryrun=False):
    """Create a loader for the test images and return the ordered list of ids."""
    root = locate_test_root(Path(data_dir))
    image_paths = find_all_image_files(root)
    if not image_paths:
        raise FileNotFoundError(f"No test images found under {root}")
    dataset = TestImageDataset(image_paths, transform=setup_center_crop_transform())
    image_ids = [p.stem for p in sorted(image_paths)]
    if dryrun:
        dataset = Subset(dataset, range(min(200, len(dataset))))
        image_ids = image_ids[: len(dataset)]
    loader = DataLoader(
        dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True
    )
    return loader, image_ids


def get_base_data_dir() -> Path:
    """Detect the base directory containing the extracted dataset."""
    candidates = [
        Path("/kaggle/input/dogs-vs-cats-redux-kernels-edition"),
        Path("/kaggle/working/dogs-vs-cats-redux-kernels-edition"),
        Path("./data/dogs-vs-cats-redux-kernels-edition"),
        Path("./dogs-vs-cats-redux-kernels-edition"),
        Path("."),
    ]
    for cand in candidates:
        if (cand / "train").exists() or (cand / "train" / "cat").exists():
            return cand
    raise FileNotFoundError("Unable to locate the dataset directory.")


base_data_dir = get_base_data_dir()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 1)  # binary output (logits)
model = model.to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
# Simple LR scheduler to improve convergence
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)




/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 99.0MB/s]


In [2]:
train_loader, val_loader = setup_train_val_loaders(
    base_data_dir, batch_size=64, dryrun=False
)

best_val_loss = float("inf")
best_state_dict = None
num_epochs = 30  # increased epochs for better performance

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0
    for imgs, targets in train_loader:
        imgs = imgs.to(device)
        targets = targets.float().unsqueeze(1).to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * imgs.size(0)
    avg_train_loss = epoch_loss / len(train_loader.dataset)
    print(f"Epoch {epoch+1}/{num_epochs} – Train loss: {avg_train_loss:.4f}")

    model.eval()
    val_losses = []
    with torch.no_grad():
        for imgs, targets in val_loader:
            imgs = imgs.to(device)
            targets = targets.float().unsqueeze(1).to(device)
            logits = model(imgs)
            loss = criterion(logits, targets)
            val_losses.append(loss.item())
    avg_val_loss = sum(val_losses) / len(val_losses)
    print(f"Epoch {epoch+1}/{num_epochs} – Val loss: {avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_state_dict = model.state_dict()

    scheduler.step()

if best_state_dict is not None:
    model.load_state_dict(best_state_dict)




FileNotFoundError: Found no valid file for the classes train. Supported extensions are: .jpg, .jpeg, .png, .ppm, .bmp, .pgm, .tif, .tiff, .webp

In [3]:
test_loader, image_ids = setup_test_loader(base_data_dir, batch_size=64, dryrun=False)
model.eval()
preds = []
with torch.no_grad():
    for imgs in test_loader:
        imgs = imgs.to(device)
        logits = model(imgs)
        probs = torch.sigmoid(logits).cpu().numpy().flatten()
        preds.extend(probs)

sample_path_candidates = list(Path(".").rglob("sample_submission.csv"))
if not sample_path_candidates:
    raise FileNotFoundError("sample_submission.csv not found in the working directory.")
sample_submission_path = sample_path_candidates[0]
sample_df = pd.read_csv(sample_submission_path, dtype={"id": str})
ordered_ids = sample_df["id"].tolist()

pred_dict = dict(zip(image_ids, preds))
ordered_preds = [pred_dict.get(str(_id), 0.5) for _id in ordered_ids]

submission = pd.DataFrame({"id": ordered_ids, "label": ordered_preds})
submission_path = Path("submission.csv")
submission.to_csv(submission_path, index=False)
print(f"Submission file saved to {submission_path.resolve()}")

Submission file saved to /kaggle/working/submission.csv
